In [ ]:
# 1. Import libraries
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

# 2. Load dataset
train = tf.keras.utils.image_dataset_from_directory(
    os.path.expanduser("~/Downloads/cats_and_dogs_filtered/train"),
    image_size=(160, 160),
    batch_size=32,   
)

val = tf.keras.utils.image_dataset_from_directory(
    os.path.expanduser("~/Downloads/cats_and_dogs_filtered/validation"),
    image_size=(160, 160),
    batch_size=32,
)

print(train.class_names)

# 3. Load pretrained MobileNetV2
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3)
)

base_model.trainable = False

# 4. Add classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

# 5. Compile model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 6. Train model
history = model.fit(
    train,
    validation_data=val,
    epochs=5
)

# 7. Evaluate model
loss, accuracy = model.evaluate(val)
print("Accuracy:", accuracy * 100, "%")

# 8. Plot accuracy
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend(["Training", "Validation"])
plt.show()

# 9. Predict new image
img = tf.keras.utils.load_img(
    os.path.expanduser("~/Downloads/cats_and_dogs_filtered/test/cat.jpg"),
    target_size=(160, 160)
)

plt.imshow(img)
plt.axis("off")
plt.show()

img = tf.keras.utils.img_to_array(img)
img = np.expand_dims(img, axis=0)
img = tf.keras.applications.mobilenet_v2.preprocess_input(img)

p = model.predict(img)

print("Dog" if p[0][0] > 0.5 else "Cat")